In [ ]:
import json
import random

random.seed(42)

dataset_with_answers_path = "musique_ans_v1.0_dev.jsonl"
examples_with_answers = []


all_lines = []
with open(dataset_with_answers_path, 'r', encoding='utf-8') as f:
    all_lines = [json.loads(line) for line in f]


examples_with_answers = random.sample(all_lines, 300)

print(f"Загружено {len(examples_with_answers)} случайных примеров")

Загружено 300 случайных примеров


In [ ]:
import pandas as pd
from typing import List, Dict, Any
import re

def transform_dataset(dataset: List[Dict[str, Any]]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Преобразует датасет в два DataFrame:
    1. paragraphs_df - параграфы с абсолютной нумерацией
    2. questions_df - подвопросы с абсолютными ID параграфов и ID общего вопроса
    """
    paragraphs_list = []
    questions_list = []
    global_paragraph_counter = 1
    paragraph_mapping = {}

    for example in dataset:
        example_id = example['id']

        for paragraph in example['paragraphs']:
            paragraph_idx = paragraph['idx']
            key = (example_id, paragraph_idx)


            paragraph_mapping[key] = global_paragraph_counter


            paragraphs_list.append({
                'absolute_id': global_paragraph_counter,
                'text': paragraph['paragraph_text']
            })

            global_paragraph_counter += 1


    for example in dataset:
        example_id = example['id']
        main_question_id = example['id']
        main_question_text = example['question']

        decomposition = example.get('question_decomposition', [])


        processed_questions = []

        for i, sub_question in enumerate(decomposition):
            question_text = sub_question['question']


            answer_mapping = {}
            for j in range(i):
                if j < len(processed_questions):
                    placeholder = f"#{j+1}"
                    answer_mapping[placeholder] = decomposition[j].get('answer', '')


            for placeholder, answer in answer_mapping.items():
                if answer:

                    #print(question_text)
                    pattern = re.escape(placeholder)
                    #print(pattern)

                    question_text = re.sub(placeholder, answer, question_text)
                    #print(question_text)

            processed_questions.append(question_text)


        for i, sub_question in enumerate(decomposition):
            paragraph_support_idx = sub_question.get('paragraph_support_idx')


            paragraph_absolute_id = None
            if paragraph_support_idx is not None:
                key = (example_id, paragraph_support_idx)
                paragraph_absolute_id = paragraph_mapping.get(key)

            questions_list.append({
                'question': processed_questions[i],
                'paragraph_absolute_id': paragraph_absolute_id,
                'main_question_id': main_question_id,
                'main_question_text': main_question_text
            })


    paragraphs_df = pd.DataFrame(paragraphs_list)
    questions_df = pd.DataFrame(questions_list)

    return paragraphs_df, questions_df

In [ ]:
paragraphs_df, questions_df = transform_dataset(examples_with_answers)
paragraphs_df

,absolute_id,text
0,1,"Bob Saget as Future Ted Mosby (voice only, unc..."
1,2,"Chai was born on April 15, 1966, in Rizhao, in..."
2,3,Tracy McConnell (colloquial: ``The Mother '') ...
3,4,"Jennifer Marie Morrison (born April 12, 1979) ..."
4,5,"The first season of How I Met Your Mother, an ..."
...,...,...
5990,5991,From 1901 to 1938 he was head of the prestigio...
5991,5992,"On the continent of Europe, Anno Domini was in..."
5992,5993,"""Beto Rockfeller"" is often mentioned as a turn..."
5993,5994,"The main representatives of the new style, oft..."


In [ ]:
import random
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

def create_retriever(qa_df, paragraphs_df, top_k=5, p_correct=0.5, seed=None, model_name='all-MiniLM-L6-v2', device='cuda'):
    """
    Создаёт функцию-имитатор ретривера с использованием эмбеддингов all-MiniLM-L6-v2.
    Правильный документ появляется в выдаче с вероятностью p_correct.
    Если правильный документ естественным образом попал в топ-k, но вероятность не сработала,
    он удаляется и заменяется следующим по релевантности неправильным.

    Параметры:
        qa_df : pd.DataFrame
            Датафрейм с колонками 'question' (текст подвопроса) и 'paragraph_absolute_id' (id правильного параграфа).
        paragraphs_df : pd.DataFrame
            Датафрейм с колонками 'id' (уникальный идентификатор) и 'text' (текст параграфа).
        top_k : int
            Количество параграфов, возвращаемых за один вызов.
        p_correct : float
            Вероятность того, что в возвращаемой порции окажется правильный документ.
        seed : int, optional
            Для воспроизводимости случайных выборов.
        model_name : str
            Название модели sentence-transformers.
        device : str
            Устройство для вычислений ('cuda' или 'cpu').

    Возвращает:
        function
            Функция retrieve(query: str) -> dict
            С полями:
                'docs': список строк вида "id. текст"
                'has_correct': bool, есть ли среди выданных документов правильный для этого вопроса
    """
    if seed is not None:
        random.seed(seed)
        torch.manual_seed(seed)


    encoder = SentenceTransformer(model_name, device=device)


    paragraph_ids = paragraphs_df['absolute_id'].tolist()
    paragraph_texts = paragraphs_df['text'].tolist()
    print("Вычисление эмбеддингов параграфов...")
    paragraph_embeddings = encoder.encode(paragraph_texts, convert_to_tensor=True, device=device, show_progress_bar=True)
    paragraph_embeddings = torch.nn.functional.normalize(paragraph_embeddings, p=2, dim=1)

    correct_map = {}
    for _, row in qa_df.iterrows():
        q = row['question']
        pid = row['paragraph_absolute_id']
        correct_map.setdefault(q, set()).add(pid)


    text_by_id = dict(zip(paragraph_ids, paragraph_texts))

    def retrieve(query, last):

        query_emb = encoder.encode(query, convert_to_tensor=True, device=device)
        query_emb = torch.nn.functional.normalize(query_emb, p=2, dim=0)


        similarities = torch.matmul(paragraph_embeddings, query_emb)
        sorted_indices = torch.argsort(similarities, descending=True).cpu().numpy()
        sorted_ids = [paragraph_ids[i] for i in sorted_indices]

        correct_ids = correct_map.get(query, set())
        #correct_ids = set(questions_df[qa_df['question'] == query]['paragraph_absolute_id'])

        # Базовый топ-k по релевантности
        candidates = sorted_ids[:top_k]
        #print('candidates',candidates)
        #print('correct_ids',correct_ids)
        if last == False:
        # Решаем, должен ли правильный присутствовать в выдаче
          include_correct = random.random() < p_correct
        else:
          include_correct = True
        #print("include_correct",include_correct)

        # Правильные id, которые уже есть в candidates
        correct_in_candidates = [pid for pid in candidates if pid in correct_ids]
        #print(correct_in_candidates)

        if include_correct:
            # Хотим, чтобы правильный был в выдаче
            if not correct_in_candidates:
                if correct_ids:
                    chosen_correct = random.choice(list(correct_ids))
                    # Заменяем случайный элемент в candidates
                    replace_idx = random.randint(0, len(candidates)-1)
                    candidates[replace_idx] = chosen_correct
        else:
            if correct_in_candidates:
                # Удаляем все правильные из candidates
                candidates_without_correct = [pid for pid in candidates if pid not in correct_ids]
                next_ids = []
                for pid in sorted_ids[top_k:]:
                    if pid not in correct_ids and pid not in candidates_without_correct:
                        next_ids.append(pid)
                    if len(candidates_without_correct) + len(next_ids) >= top_k:
                        break
                # Дополняем до top_k
                candidates = (candidates_without_correct + next_ids)[:top_k]


        random.shuffle(candidates)

        docs = [f"{pid}. {text_by_id[pid]}" for pid in candidates]
        ids = [pid for pid in candidates]

        return {
            'docs': docs,
            'ids': ids
        }

    return retrieve

In [ ]:
TOP_K = 5
P_CORRECT = 0.3 #0.3
SEED = 42
RETRIEVER_MODEL = 'all-MiniLM-L6-v2'
DEVICE = "cuda"

retriever = create_retriever(questions_df, paragraphs_df, top_k=TOP_K, p_correct=P_CORRECT,
                             seed=SEED, model_name=RETRIEVER_MODEL, device=DEVICE)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6340.46it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Вычисление эмбеддингов параграфов...


Batches: 100%|██████████| 188/188 [00:02<00:00, 72.18it/s]


In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "INDEX_SEARCH_TOOL",
            "description": "Retrieve documents from the index by a search query.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "submit_answer",
            "description": "Submit the final answer with the document ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer", "description": "Document ID that best answers the question"}
                },
                "required": ["id"]
            }
        }
    }
]

In [ ]:
import json
import re
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from typing import List, Dict, Any, Optional, Tuple, Union


class QwenAgent:
    def __init__(self, model_name: str = "Qwen/Qwen2.5-7B-Instruct", device_map: str = "auto"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16,
            device_map=device_map
        )
        self.model.eval()

    def _call_model(self, messages: List[Dict[str, Any]], max_new_tokens: int = 512):
        """
        Вызывает модель и возвращает сгенерированный текст, ID токенов и логиты.
        Returns:
            (response, generated_ids, scores)
            response: str
            generated_ids: torch.Tensor (1, num_generated)
            scores: tuple of torch.Tensor (num_generated, 1, vocab_size)
        """
        prompt = self.tokenizer.apply_chat_template(
            messages,
            tools=TOOLS,
            add_generation_prompt=True,
            tokenize=False
        )
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0,
                do_sample=False,
                return_dict_in_generate=True,
                output_scores=True,
            )
        # generated_ids: [1, total_seq_len]; входные + новые
        total_ids = outputs.sequences[0]
        input_len = inputs.input_ids.shape[1]
        generated_ids = total_ids[input_len:]  # только новые токены
        scores = outputs.scores  # tuple of tensors, каждый [1, vocab_size]
        response = self.tokenizer.decode(generated_ids, skip_special_tokens=False)
        return response, generated_ids, scores

    def _extract_tool_call(self, response: str) -> Optional[Dict[str, Any]]:
        # 1. Пробуем JSON
        json_pattern = r'(<tool_call>.*?<|im_end|>)'#r'<tool_call>(.*?)</tool_call>'
        match = re.search(json_pattern, response, re.DOTALL)
        print(repr(response))
        #jj = json.loads(match.group(1))
        print("MATCH", match)
        if match:
            try:
                return json.loads(match.group(1))
            except json.JSONDecodeError:
                pass

        # 2. Пробуем XML: <function=NAME><parameter=ARG>value</parameter></function>
        xml_pattern = r'<tool_call>.*?<function=(\w+)>.*?<parameter=(\w+)>(.*?)</parameter>.*?</tool_call>'
        match = re.search(xml_pattern, response, re.DOTALL)
        if match:
            name = match.group(1)
            param_name = match.group(2)
            param_value = match.group(3).strip()
            # Убираем возможные кавычки вокруг строки
            if param_value.startswith('"') and param_value.endswith('"'):
                param_value = param_value[1:-1]
            return {"name": name, "arguments": {param_name: param_value}}

        return None

    def _compute_tool_metric(self, response: str, generated_ids: torch.Tensor, scores: tuple, metric: str = 'entropy') -> Optional[float]:
        """
        Вычисляет среднюю энтропию (или NLL) для токенов, образующих вызов инструмента.
        Аргументы:
            response: полный сгенерированный текст (без входа)
            generated_ids: тензор ID сгенерированных токенов (1, num_tokens)
            scores: кортеж тензоров логитов для каждого шага
            metric: 'entropy' или 'nll'
        Returns:
            среднее значение метрики по токенам вызова, или None, если вызов не найден
        """
        # Ищем в ответе подстроку вызова инструмента (включая теги)
        tool_call_pattern =r'(<tool_call>.*?<|im_end|>)'#r'(<tool_call>.*?</tool_call>)'
        match = re.search(tool_call_pattern, response, re.DOTALL)
        if not match:
            return None
        tool_call_str = match.group(1)
        start_idx = match.start()
        end_idx = match.end()

        # Декодируем каждый токен по отдельности для позиций
        tokens = []
        positions = []
        pos = 0
        for tid in generated_ids:
            token_str = self.tokenizer.decode([tid], skip_special_tokens=False)
            tokens.append(token_str)
            positions.append(pos)
            pos += len(token_str)

        full_text = ''.join(tokens)


        # Определяем индексы токенов, перекрывающихся с tool_call_str
        indices = []
        for i, (token, p) in enumerate(zip(tokens, positions)):
            token_end = p + len(token)
            if token_end > start_idx and p < end_idx:
                indices.append(i)

        if not indices:
            return None

        values = []
        for i in indices:
            logits = scores[i]          # [1, vocab_size]
            logp = F.log_softmax(logits, dim=-1)   # [1, vocab_size]
            if metric == 'entropy':
                p = logp.exp()
                entropy = -(p * logp).sum(dim=-1).item()
                values.append(entropy)
            elif metric == 'nll':
                token_id = generated_ids[i].item()
                nll = -logp[0, token_id].item()
                values.append(nll)
            else:
                raise ValueError("metric must be 'entropy' or 'nll'")

        return sum(values) / len(values)

    def step(self, messages: List[Dict[str, Any]], metric: str = 'entropy') -> Tuple[Optional[Dict[str, Any]], str, Optional[float]]:
        """
        Выполняет один шаг: вызывает модель, извлекает вызов инструмента и вычисляет метрику для токенов вызова.
        Args:
            messages: история сообщений
            metric: 'entropy' или 'nll' – тип вычисляемой метрики
        Returns:
            (tool_call, raw_response, metric_value)
            tool_call: словарь с name и arguments, если найден, иначе None
            raw_response: полный текст ответа модели
            metric_value: средняя энтропия (или NLL) для токенов вызова, или None, если вызов не найден
        """
        response, gen_ids, scores = self._call_model(messages)
        tool_call = self._extract_tool_call(response)
        tool_metric = self._compute_tool_metric(response, gen_ids, scores, metric) if tool_call else None
        return tool_call, response, tool_metric

In [ ]:
import re
import torch
import torch.nn.functional as F

# ======================== ПРОМПТ АГЕНТА ========================
SYSTEM_PROMPT = """
ROLE:
You are a precise knowledge assistant. Answer the user's question using the provided documents.

RULES:
1. If documents contain the answer, select the most explicit and complete document and call `submit_answer` with its id.
2. If no document clearly answers the question, call `INDEX_SEARCH_TOOL` to retrieve more relevant documents. You may reformulate the query.
3. Do not call `INDEX_SEARCH_TOOL` repeatedly with semantically identical queries.
4. Never invent document ids or content. Only rely on tool-provided observations.

FORMAT:
- Before each tool call, you may provide a short thought (1‑2 sentences, <40 words) inside `<thought>` tags.
- Your final response must be a tool call in the following format:
  `<tool_call>{"name": "<tool_name>", "arguments": {...}}</tool_call>`
- Be sure to add the beginning <tool_call> and end </tool_call> of the tool call
- Do not output `<observation>` or document text; the system will provide them after your tool call.

EXAMPLES:

EXAMPLE 1 (answer not in initial documents):
question: Who is the current president of France?
Documents:
31. The French President is elected for a five-year term.
222. The Élysée Palace is the official residence of the President.
413. France has a semi-presidential system.
<thought>These documents give background but don't name the current president. I need a more specific query.</thought>
<tool_call>{"name": "INDEX_SEARCH_TOOL", "arguments": {"query": "Who is the current president of France"}}</tool_call>
<observation> (system provides)
6. Emmanuel Macron was re-elected in 2022.
79. Emmanuel Macron is the current president of France.
</observation>
<thought>Document 79 explicitly names Emmanuel Macron as current president.</thought>
<tool_call>{"name": "submit_answer", "arguments": {"id": 79}}</tool_call>

EXAMPLE 2 (answer found after tool call):
question: When did the Berlin Wall fall?
Documents:
45. The Berlin Wall divided Berlin from 1961 to 1989.
12. The fall paved the way for German reunification.
<thought>The exact date is missing. I'll search for it.</thought>
<tool_call>{"name": "INDEX_SEARCH_TOOL", "arguments": {"query": "Berlin Wall fall date"}}</tool_call>
<observation>
23. The Berlin Wall fell on November 9, 1989.
34. November 9, 1989 is a historic date.
</observation>
<thought>Document 23 gives the exact date.</thought>
<tool_call>{"name": "submit_answer", "arguments": {"id": 23}}</tool_call>

Now begin."""

In [ ]:
llm = QwenAgent(model_name="Qwen/Qwen2.5-7B-Instruct")

Loading weights: 100%|██████████| 339/339 [00:02<00:00, 118.55it/s]
Some parameters are on the meta device because they were offloaded to the cpu.


In [ ]:
# def get_tool_cross_entropy(logprobs_content, tool_name="INDEX_SEARCH_TOOL", use_negative=True):
#     if not logprobs_content:
#         return None, None, None

#     def get_token_and_logprob(item):
#         if hasattr(item, 'token') and hasattr(item, 'logprob'):
#             return item.token, item.logprob
#         elif isinstance(item, dict):
#             return item.get('token', ''), item.get('logprob', 0.0)
#         else:
#             raise TypeError("Элемент должен иметь атрибуты token и logprob")

#     tokens = []
#     logprobs = []
#     positions = []
#     current_pos = 0

#     for item in logprobs_content:
#         token, logprob = get_token_and_logprob(item)
#         tokens.append(token)
#         logprobs.append(logprob)
#         positions.append(current_pos)
#         current_pos += len(token)

#     full_text = ''.join(tokens)
#     print(f"FULL_TEXT: {full_text}")

#     start_idx = full_text.find(tool_name)
#     print(f"START_INDEX: {start_idx}")
#     if start_idx == -1:
#         return None, None, None

#     end_idx = start_idx + len(tool_name)

#     entropy_values = []
#     matched_tokens = []

#     for i, token in enumerate(tokens):
#         token_start = positions[i]
#         token_end = token_start + len(token)
#         if token_end > start_idx and token_start < end_idx:
#             logp = logprobs[i]
#             value = -logp if use_negative else logp
#             entropy_values.append(value)
#             matched_tokens.append(token)

#     if not entropy_values:
#         return None, None, None

#     avg_entropy = sum(entropy_values) / len(entropy_values)
#     return entropy_values, avg_entropy, matched_tokens

In [ ]:
def index_search_tool(query, last):
    result = retriever(query, last)   # result['docs'] — список строк документов
    tool_content = "\n".join(result['docs'])   # просто текст, без обёртки
    return result['ids'], tool_content

In [ ]:
def run_agent_for_subquestion(agent, subquestion, initial_docs, ids, correct_ids, max_calls=5, max_steps=5):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"question: {subquestion}\n\nDocuments:\n{initial_docs}"}
    ]
    tool_calls_used = 0
    step_info_history = []
    final_answer_id = None
    success = False

    for step in range(max_steps):
        tool_call, response, metric = agent.step(messages, metric='entropy')
        print(f'Step {step+1} response:\n{response}')
        if metric is not None:
            print(f"Tool call metric (entropy): {metric:.4f}")

        if not tool_call:
            print("No valid tool call found. Stopping.")
            break

        name = tool_call.get("name")
        args = tool_call.get("arguments", {})

        has_correct = len(set(ids) & correct_ids) > 0

        step_info = {
            "step": step,
            "action": f"{name}({json.dumps(args)})",
            "has_correct": has_correct,
            "entropy": metric,
            #"entropy_list": None,
            "text": response,
        }
        step_info_history.append(step_info)

        messages.append({"role": "assistant", "content": response})

        if name == "submit_answer":
            final_answer_id = args.get("id")
            success = True
            break

        elif name == "INDEX_SEARCH_TOOL":
            if tool_calls_used >= max_calls:
                print("Max tool calls exceeded, stopping.")
                break
            query = args.get("query", "")
            last = (step == max_steps - 1)
            ids, tool_content = index_search_tool(query, last)
            has_correct = len(set(ids) & set(correct_ids)) > 0
            #step_info["has_correct"] = has_correct
            print(f"Tool result (has_correct={has_correct}):\n{tool_content}")
            messages.append({"role": "tool", "name": name, "content": tool_content})
            tool_calls_used += 1
            continue
        else:
            print(f"Unknown tool: {name}")
            break

    if final_answer_id is None:
        final_answer_id = 'No answer'
    return final_answer_id, step_info_history, success

In [ ]:
def make_serializable(obj):
    if isinstance(obj, set):
        return list(obj)
    if hasattr(obj, 'tolist'):   # для тензоров PyTorch / TensorFlow / NumPy
        return obj.tolist()
    if isinstance(obj, dict):
        return {k: make_serializable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [make_serializable(item) for item in obj]
    return obj

In [ ]:
from tqdm import tqdm
import json

unique_questions = questions_df['question']  # предполагается, что уже уникальные
results = {}

MAX_TOOL_CALLS = 5
SAVE_PATH = 'results_qwen_3_5_4B_ce.json'

for subq in tqdm(unique_questions):
    # Получаем начальную выдачу
    first_page = retriever(subq, False)
    ids = first_page['ids']
    initial_docs_str = "\n".join(first_page['docs'])
    correct_ids = set(questions_df[questions_df['question'] == subq]['paragraph_absolute_id'])
    print('IDS:', ids)
    print('correct_ids:', correct_ids)

    has_correct = len(set(ids) & correct_ids) > 0
    print("INITIAL has_correct:",has_correct)

    try:
        doc_id, step_history, success = run_agent_for_subquestion(
            ids = ids,
            agent=llm,
            subquestion=subq,
            initial_docs=initial_docs_str,
            correct_ids=correct_ids,
            max_calls=MAX_TOOL_CALLS,
            max_steps=5
        )
    except Exception as e:
        print(f"Error for subquestion '{subq}': {e}")
        doc_id, step_history, success = None, None, False
    try:
        doc_id = int(doc_id)
    except:
        doc_id = doc_id
    is_correct = doc_id in correct_ids if doc_id != 'No answer' else False

    results[subq] = {
        "found_doc": doc_id,
        "correct_ids": list(correct_ids),
        "is_correct": is_correct,
        "logits_history": step_history,
        "initial_has_correct": has_correct,
        "success": success
    }

    print(f"Агент вернул: {doc_id} (правильные: {correct_ids}) -> {'✓' if is_correct else '✗'}")
    print("="*50)

    # Сохраняем промежуточные результаты
    serializable_results = make_serializable(results)   # функция должна быть определена
    with open(SAVE_PATH, 'w', encoding='utf-8') as f:
        json.dump(serializable_results, f, ensure_ascii=False, indent=2)
    print(f"Промежуточные результаты сохранены: {SAVE_PATH}")

  0%|          | 0/778 [00:00<?, ?it/s]

IDS: [2822, 2820, 5, 7, 2823]
correct_ids: {17}
INITIAL has_correct: False


  0%|          | 1/778 [00:22<4:49:09, 22.33s/it]

'<translation>\n{"name": "submit_answer", "arguments": {"id": 2820}}\n</tool_call><|im_end|>'
MATCH <re.Match object; span=(12, 13), match='>'>
Step 1 response:
<translation>
{"name": "submit_answer", "arguments": {"id": 2820}}
</tool_call><|im_end|>
No valid tool call found. Stopping.
Агент вернул: No answer (правильные: {17}) -> ✗
Промежуточные результаты сохранены: results_qwen_2_5_7B_Instruct_ce_new.json
IDS: [2825, 7, 3, 2820, 8]
correct_ids: {3}
INITIAL has_correct: True


  0%|          | 2/778 [01:07<7:42:09, 35.73s/it]

...

'<thought>None of the provided documents mention the Somali Muslim Ajuran Empire or any proclamations of independence. I need to search for more relevant information.</thought>\n<tool_call>{"name": "INDEX_SEARCH_TOOL", "arguments": {"query": "Somali Muslim Ajuran Empire proclamation of independence"}}<|im_end|>'
MATCH <re.Match object; span=(8, 9), match='>'>
Step 1 response:
<thought>None of the provided documents mention the Somali Muslim Ajuran Empire or any proclamations of independence. I need to search for more relevant information.</thought>
<tool_call>{"name": "INDEX_SEARCH_TOOL", "arguments": {"query": "Somali Muslim Ajuran Empire proclamation of independence"}}<|im_end|>
No valid tool call found. Stopping.
Агент вернул: No answer (правильные: {2784, 466}) -> ✗
Промежуточные результаты сохранены: results_qwen_2_5_7B_Instruct_ce_new.json
IDS: [4983, 3775, 4978, 3769, 4989]
correct_ids: {465, 2780}
INITIAL has_correct: False


  7%|▋         | 55/778 [41:52<9:41:32, 48.26s/it]

'<thought>The provided documents do not mention the expulsion of Portuguese from Myanmar. I will use the INDEX_SEARCH_TOOL to find more relevant information.</thought>\n<tool_call>{"name": "INDEX_SEARCH_TOOL", "arguments": {"query": "Portuguese expelled from Myanmar"}}<|im_end|>'
MATCH <re.Match object; span=(8, 9), match='>'>
Step 1 response:
<thought>The provided documents do not mention the expulsion of Portuguese from Myanmar. I will use the INDEX_SEARCH_TOOL to find more relevant information.</thought>
<tool_call>{"name": "INDEX_SEARCH_TOOL", "arguments": {"query": "Portuguese expelled from Myanmar"}}<|im_end|>
No valid tool call found. Stopping.
Агент вернул: No answer (правильные: {465, 2780}) -> ✗
Промежуточные результаты сохранены: results_qwen_2_5_7B_Instruct_ce_new.json
IDS: [60, 53, 55, 734, 50]
correct_ids: {492}
INITIAL has_correct: False


  7%|▋         | 56/778 [42:35<9:21:39, 46.67s/it]

'<thought>Document 53 mentions Tom Denney and his place of birth, which is what the question is asking for.</thought>\n<tool_call>{"name": "submit_answer", "arguments": {"id": 53}}<|im_end|>'
MATCH <re.Match object; span=(8, 9), match='>'>
Step 1 response:
<thought>Document 53 mentions Tom Denney and his place of birth, which is what the question is asking for.</thought>
<tool_call>{"name": "submit_answer", "arguments": {"id": 53}}<|im_end|>
No valid tool call found. Stopping.
Агент вернул: No answer (правильные: {492}) -> ✗
Промежуточные результаты сохранены: results_qwen_2_5_7B_Instruct_ce_new.json
IDS: [480, 590, 5098, 42, 481]
correct_ids: {480, 42, 590}
INITIAL has_correct: True


In [ ]:
print("\n" + "="*50)
print("ИТОГИ ЭКСПЕРИМЕНТА")
for subq, res in results.items():
  print(f"{subq[:50]}...: вернул {res['found_doc']} (правильно: {res['is_correct']}), "
        f"нач.выдача содержала ответ: {res['initial_has_correct']}")

In [ ]:
1+1